# Sprint 1 — Analisi Preliminare

> 📊 **Dataset**: UCI Machine Learning Repository — *Predict Students'
> Dropout and Academic Success* (4.424 studenti, 36 feature + 1 target).
> 🔗 https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

## Obiettivo del notebook

Questo notebook copre le **User Story di Sprint 1** del progetto di
predizione del dropout universitario. Lo scopo dello Sprint è preparare
il dataset prima dell'analisi esplorativa: caricamento, definizione del
target binario, normalizzazione dei nomi colonna, gestione di duplicati
e missing values, prime statistiche descrittive.

In altre parole: alla fine di questo notebook avremo un DataFrame pulito,
ben tipizzato e con un target binario `Dropout` / `Graduate`, pronto per
le US di Sprint 2 (analisi esplorativa) e Sprint 3 (modeling).

## Indice e responsabilità

| # | User Story | Responsabile |
|---|---|---|
| 1 | [US-01 · Caricamento e prima ispezione del dataset](#us-01) | Youssra |
| 2 | [US-02 · Rimozione "Enrolled" e definizione del target](#us-02) | Jessica |
| 3 | [US-03 · Rinomina delle variabili](#us-03) | Youssra |
| 4 | [US-04 · Controllo dei duplicati](#us-04) | Jessica |
| 5 | [US-05 · Analisi dei missing values](#us-05) | Youssra |
| 6 | [US-06 · Analisi descrittiva delle variabili numeriche](#us-06) | Jessica |

## Note sull'esecuzione

- **Path del dataset**: `../data/raw/data.csv` (relativo alla cartella
  `notebooks/`). Il file usa `;` come separatore, non la virgola.
- **Esecuzione lineare**: le celle vanno eseguite in ordine, dall'alto
  verso il basso. Ogni US assume che le precedenti siano state eseguite.
- **Output dei grafici**: salvati in `outputs/figures/` con naming
  `usXX_<descrizione-kebab-case>.png`.
- **Funzioni riutilizzabili**: importate da `src/` quando applicabile.

---

## Setup

Dipendenze del notebook. Le librerie sono elencate in `requirements.txt`
e vanno installate nel virtual env del progetto prima di eseguire le
celle successive.

In [ ]:
# Standard library
import os
import sys

# Aggiungi la root del progetto al path Python, così che `src` sia importabile
sys.path.insert(0, os.path.abspath('..'))

# Third-party
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Local
from src.preprocessing import rename_columns

---

## US-01 · Caricamento e Prima Ispezione del Dataset

**Responsabile**: Youssra Zarouky

In questa sezione carichiamo il dataset e verifichiamo che sia quello corretto 
prima di fare qualsiasi analisi.

In [ ]:
df = pd.read_csv('../data/raw/data.csv', sep=';')

df.head()

### Verifica dimensioni e tipi di dato

Controlliamo che il numero di righe e colonne corrisponda a quello atteso dalla documentazione UCI, 
e che i tipi di dato siano corretti.

In [ ]:
# Il dataset dovrebbe avere 4424 righe e 37 colonne (36 feature + 1 target) come da documentazione UCI
print("Righe e colonne:", df.shape)
print()
print(df.dtypes)
# La maggior parte delle variabili sono numeriche (int64, float64)
# Alcune variabili categoriche (es. stato civile, corso) sono codificate come int
# Solo Target è testuale (object): contiene Dropout, Graduate, Enrolled

In [ ]:
df.tail()

---

## US-02 · Rimozione "Enrolled" e Definizione del Target <a id="us-02"></a>

**User Story**: come Ufficio per il Successo Accademico, vogliamo che il
dataset venga filtrato per escludere gli studenti ancora iscritti, affinché
il modello lavori solo su outcome definitivi e il problema sia ben definito.

**Responsabile**: Jessica M Pucci

### Obiettivo
Filtrare il dataset per tenere solo gli studenti con outcome accademico
definitivo (`Dropout` o `Graduate`), escludendo gli studenti ancora iscritti
(`Enrolled`) il cui percorso non è ancora concluso. Codificare il target
in formato numerico per le fasi successive di modeling.

### Approccio
1. **Snapshot della distribuzione iniziale** del target a 3 classi
2. **Filtro** delle righe con `Target == 'Enrolled'`
3. **Encoding** del target binario (`Dropout=1, Graduate=0`)
4. **Snapshot della distribuzione finale** a 2 classi
5. **Visualizzazione comparativa** prima/dopo per la presentazione finale

### 2.1 Snapshot della distribuzione iniziale

Cattura la distribuzione del target a 3 classi (`Dropout`, `Graduate`,
`Enrolled`) prima del filtro, in valori assoluti e percentuali. Sarà la
base per documentare l'impatto del filtro e per il grafico comparativo
finale.

In [ ]:
# Snapshot della distribuzione del target PRIMA del filtro.
# Costruiamo una tabella con conteggi assoluti e proporzioni,
# per documentare lo stato iniziale e calibrare il grafico comparativo.

# Conteggi assoluti per ciascuna classe del target
counts_before = df['Target'].value_counts()

# Proporzioni relative (somma = 1.0)
proportions_before = df['Target'].value_counts(normalize=True)

# Combiniamo in un unico DataFrame per leggibilità
distribution_before = pd.DataFrame({
    'count': counts_before,
    'percentage': (proportions_before * 100).round(2),
})

# Aggiungiamo una riga di totale per dare contesto
distribution_before.loc['TOTAL'] = [len(df), 100.00]

print("Distribuzione del target — prima del filtro:")
distribution_before

### 2.2 Filtro degli studenti "Enrolled"

Rimuoviamo dal dataset gli studenti con outcome non ancora definitivo
(`Target == 'Enrolled'`), mantenendo solo i casi `Dropout` e `Graduate`.
Resettiamo l'indice del DataFrame per evitare buchi di numerazione.

In [ ]:
# Salviamo la dimensione di partenza per il log "prima/dopo".
# Senza questa cattura prima del filtro perdiamo il riferimento.
n_before = len(df)

# Filtro: teniamo solo gli studenti con outcome definitivo.
# La condizione "df['Target'] != 'Enrolled'" produce una Serie booleana
# che pandas usa per selezionare le righe in cui la condizione è True.
df = df[df['Target'] != 'Enrolled']

# Reset dell'indice: dopo il filtro gli indici sono "bucati"
# (es. 0, 1, 3, 5, ...). Li rigeneriamo contigui (0, 1, 2, ...).
# drop=True evita che i vecchi indici vengano salvati come colonna.
df = df.reset_index(drop=True)

# Log prima/dopo: documenta l'impatto del filtro nell'output del notebook
n_after = len(df)
print(f"Righe prima del filtro: {n_before}")
print(f"Righe dopo il filtro:   {n_after}")
print(f"Studenti Enrolled rimossi: {n_before - n_after}")

### 2.3 Encoding del target

Convertiamo il target da stringa (`'Dropout'`, `'Graduate'`) a intero
(`1`, `0`) per renderlo digeribile dagli algoritmi di scikit-learn
nelle fasi successive di modeling.

#### Una scelta consapevole: `Dropout=1, Graduate=0`

Il brief originale suggerisce `Dropout=0, Graduate=1`. Adottiamo invece
la convenzione inversa: `Dropout=1, Graduate=0`.

In machine learning la "classe positiva" (codificata `1`) è per
convenzione la classe di interesse del problema. Nel nostro caso la
classe di interesse sono gli studenti a rischio dropout — è proprio
quella che vogliamo identificare. Codificarli come `1` rende le metriche
standard di scikit-learn (Precision, Recall, F1) direttamente leggibili
come "qualità della predizione del dropout", senza dover ogni volta
specificare `pos_label=0` o ragionare al contrario sulle probabilità.

Lo stesso ragionamento si trova in ambito medico, dove "test positivo"
indica per convenzione la presenza della condizione che si sta cercando.

In [ ]:
# Encoding del target: Dropout=1, Graduate=0.
# Convenzione ML: la classe positiva (1) è la classe di interesse del
# problema (gli studenti a rischio dropout). Questo semplifica la lettura
# delle metriche in Sprint 3 (Recall, Precision, F1 della classe 1
# saranno direttamente le metriche sulla classe da identificare).
# Scelta che si discosta dal brief originale (che suggeriva Dropout=0):
# documentata esplicitamente più sotto nella cella narrativa.

target_mapping = {'Dropout': 1, 'Graduate': 0}
df['Target'] = df['Target'].map(target_mapping)

# Verifica veloce: il dtype dovrebbe essere ora int64,
# e i valori unici devono essere {0, 1} senza NaN.
print(f"Tipo dato Target: {df['Target'].dtype}")
print(f"Valori unici:     {sorted(df['Target'].unique())}")
print(f"NaN nel target:   {df['Target'].isna().sum()}")

### 2.4 Snapshot della distribuzione finale

Cattura la distribuzione del target dopo il filtro e l'encoding, ora
binaria (`Dropout=1`, `Graduate=0`). Specchio strutturale dello snapshot
iniziale: stessi calcoli, stesso formato, su un DataFrame trasformato.

In [ ]:
# Snapshot della distribuzione del target DOPO il filtro e l'encoding.
# Specchio strutturale di "distribution_before": stessi calcoli, stesso
# formato, su DataFrame ora binario (Dropout=1, Graduate=0).

# Conteggi assoluti e proporzioni sul df filtrato
counts_after = df['Target'].value_counts()
proportions_after = df['Target'].value_counts(normalize=True)

# Combiniamo in DataFrame, identico per struttura a distribution_before
distribution_after = pd.DataFrame({
    'count': counts_after,
    'percentage': (proportions_after * 100).round(2),
})

# Rinomina degli indici per leggibilità: 0/1 → label originali.
# Il df mantiene gli interi; la rinomina è solo cosmetica per l'output.
distribution_after = distribution_after.rename(index={0: 'Graduate (0)', 1: 'Dropout (1)'})

# Riga di totale, come per lo snapshot iniziale
distribution_after.loc['TOTAL'] = [len(df), 100.00]

print("Distribuzione del target — dopo il filtro e l'encoding:")
distribution_after

### 2.5 Visualizzazione comparativa prima/dopo

Bar chart con le tre classi originali del target. Le due classi
mantenute (`Dropout`, `Graduate`) sono colorate per dare loro
identità; la classe rimossa (`Enrolled`) è in grigio neutro per
comunicare visivamente la scelta metodologica del filtro. Sopra ogni
barra: conteggio assoluto e percentuale sul totale di partenza.

In [ ]:
# ====================================================================
# Grafico comparativo PRIMA / DOPO della rimozione degli Enrolled
# ====================================================================

# === STEP A: Funzione di disegno riusabile ===

def draw_target_bars(ax, data, total, palette):
    """
    Disegna un bar chart del target su un asse, con annotazioni e stile.

    Parametri
    ---------
    ax : matplotlib.axes.Axes
        L'asse su cui disegnare.
    data : pd.DataFrame
        DataFrame con indice = nomi classi e colonna 'count'.
    total : int
        Totale di riferimento per il calcolo delle percentuali.
    palette : dict
        Mapping {nome_classe: colore_hex}.
    """
    # --- Disegno barre ---
    sns.barplot(
        x=data.index,
        y=data['count'],
        hue=data.index,
        palette=palette,
        saturation=1.0,
        edgecolor='none',
        legend=False,
        ax=ax,
    )

    # Hatching diagonale solo sulla barra Enrolled (se presente)
    for patch, label in zip(ax.patches, data.index):
        if label == 'Enrolled':
            patch.set_hatch('///')
            patch.set_edgecolor('#666666')

    # --- Annotazioni numeriche sopra le barre ---
    y_max = data['count'].max()
    text_offset = y_max * 0.02

    for patch, label in zip(ax.patches, data.index):
        count = data.loc[label, 'count']
        # Percentuale ricalcolata sul totale passato (PRIMA o DOPO)
        pct = (count / total) * 100

        x_pos = patch.get_x() + patch.get_width() / 2
        y_pos = patch.get_height() + text_offset

        # Formato numerico stile italiano: punto migliaia, virgola decimali
        count_text = f"{int(count):,}".replace(",", ".")
        pct_text = f"{pct:.2f}".replace(".", ",") + "%"

        annotation = f"{count_text}\n{pct_text}"

        ax.text(
            x_pos, y_pos, annotation,
            ha='center',
            va='bottom',
            fontsize=11,
            color='#333333',
            linespacing=1.3,
        )

    # --- Rifiniture asse ---
    ax.set_ylim(0, y_max * 1.18)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    ax.set_xlabel('')
    ax.tick_params(axis='both', labelsize=10, colors='#333333')


# === STEP B: Preparazione dei due dataset ===

# Palette: convenzioni del progetto (US-07) + grigio chiaro per Enrolled
class_colors = {
    'Dropout':  '#1e90ff',   # Dodger Blue
    'Graduate': '#07004d',   # Navy
    'Enrolled': '#d8d8d8',   # grigio chiaro: classe rimossa
}

# --- Dataset PRIMA: dallo snapshot iniziale ---
plot_data_before = distribution_before.drop('TOTAL').copy()
plot_data_before = plot_data_before.reindex(['Dropout', 'Graduate', 'Enrolled'])
total_before = int(distribution_before.loc['TOTAL', 'count'])

# --- Dataset DOPO: ricostruito dal df filtrato ---
# Nel df attuale Target è codificato (0=Graduate, 1=Dropout).
plot_data_after = pd.DataFrame({
    'count': [
        (df['Target'] == 1).sum(),   # Dropout
        (df['Target'] == 0).sum(),   # Graduate
    ]
}, index=['Dropout', 'Graduate'])
total_after = len(df)


# === STEP C: Figura, subplot, titoli ===

# Figura con due subplot affiancati di larghezza identica
fig, (ax_before, ax_after) = plt.subplots(
    1, 2,
    figsize=(14, 6),
    sharey=True,
    gridspec_kw={'width_ratios': [1, 1]},   # larghezze identiche
)

# Disegno dei due grafici tramite la funzione
draw_target_bars(ax_before, plot_data_before, total_before, class_colors)
draw_target_bars(ax_after,  plot_data_after,  total_after,  class_colors)

# Label asse Y su entrambi i subplot
ax_before.set_ylabel('Numero di studenti', fontsize=11)
ax_after.set_ylabel('Numero di studenti', fontsize=11)

# Riabilita i tick label sull'asse Y del subplot di destra
# (sharey li nasconderebbe di default)
ax_after.tick_params(axis='y', labelleft=True)

# Mini-titoli sopra ciascun subplot
ax_before.set_title(
    f"PRIMA — {total_before:,} studenti".replace(",", "."),
    fontsize=12,
    color='#333333',
    loc='left',
    pad=10,
)
ax_after.set_title(
    f"DOPO — {total_after:,} studenti".replace(",", "."),
    fontsize=12,
    color='#333333',
    loc='left',
    pad=10,
)

# Titolo + sottotitolo dell'intera figura, ancorati all'asse di sinistra
axis_left = ax_before.get_position().x0

fig.text(
    axis_left, 0.961,
    'Da 3 classi a 2: la rimozione degli studenti Enrolled',
    fontsize=15,
    fontweight='bold',
    ha='left',
    va='top',
)

fig.text(
    axis_left, 0.92,
    'Gli studenti ancora iscritti hanno outcome non definitivo: '
    'esclusi dall\'analisi.',
    fontsize=11,
    fontstyle='italic',
    color='#555555',
    ha='left',
    va='top',
)

# Spazio in cima per i titoli
plt.subplots_adjust(top=0.82)

plt.show()

### 2.6 Salvataggio della figura

Salviamo il PNG in `outputs/figures/` per disporre dell'asset come
file indipendente, utilizzabile nella presentazione finale.
Le convenzioni di naming, dpi e bbox sono quelle definite in
`PROJECT_CONVENTIONS.md`.

In [ ]:
# === Salvataggio della figura ===

# Path di destinazione: cartella outputs/figures/ relativa alla root del progetto
output_dir = '../outputs/figures'
output_filename = 'us02_target_distribution_before_after.png'
output_path = os.path.join(output_dir, output_filename)

# Creiamo la cartella se non esiste (idempotente: non fa nulla se c'è già)
os.makedirs(output_dir, exist_ok=True)

# Salvataggio: dpi 150 e bbox tight come da convenzioni del progetto
fig.savefig(
    output_path,
    dpi=150,
    bbox_inches='tight',
)

print(f"Figura salvata in: {output_path}")

### Risultato

Da un dataset iniziale di **4.424 studenti** ripartiti su 3 classi
(`Dropout`, `Graduate`, `Enrolled`), abbiamo escluso i **794 studenti
ancora iscritti** (17,95% del totale) e codificato il target binario
risultante. Il dataset finale conta **3.630 studenti** suddivisi in:

- **Graduate** (codificato `0`): 2.209 studenti, 60,85% del nuovo totale
- **Dropout** (codificato `1`): 1.421 studenti, 39,15% del nuovo totale

### Considerazione metodologica

L'esclusione degli studenti `Enrolled` non è una pulizia di dati
"sporchi" ma una scelta di **definizione del problema**. Gli iscritti
hanno un percorso ancora aperto: includerli avrebbe significato
trasformare la classificazione binaria in un problema multi-classe con
una classe semanticamente diversa dalle altre due — un outcome
"sospeso" accanto a due outcome conclusi. Limitare l'analisi ai casi
con outcome definitivo permette al modello di apprendere segnali
chiari e di restituire predizioni interpretabili.

La distribuzione finale presenta uno **sbilanciamento moderato**
(60,85% Graduate vs 39,15% Dropout). Non è drammatico — un modello
banale che predicesse sempre la classe maggioritaria avrebbe accuracy
del 60,85%, già "sospettoso" ma non bloccante. Va comunque tenuto
presente: in Sprint 3 dovremo usare metriche oltre l'accuracy
(F1, Precision, Recall) e considerare strategie di compensazione
(`class_weight='balanced'` nei modelli, eventuale resampling).

### Implicazioni per le US successive

- **US-03 (rinomina variabili)**: lavorerà sul dataset filtrato di
  3.630 righe, non sul dataset originale. La colonna target attualmente
  si chiama `Target` (PascalCase) e verrà ridenominata a `target` con
  il resto delle colonne
- **US-04 (controllo duplicati)**: il controllo va fatto sul dataset
  post-filtro, dove la probabilità di duplicati esatti è bassa ma il
  test resta una garanzia metodologica
- **Sprint 3 (modeling)**: lo sbilanciamento 60,85 / 39,15 va gestito
  esplicitamente. La classe positiva del problema è codificata `1`
  (Dropout): tutte le metriche di scikit-learn riferite alla classe
  positiva saranno direttamente interpretabili come "qualità della
  predizione del rischio dropout"

---

## US-03 · Rinomina delle Variabili

In questa sezione rinominiamo tutte le colonne del dataset in `snake_case`,
rimuovendo spazi, apostrofi e caratteri speciali.
La funzione è stata estratta in `src/preprocessing.py` per riutilizzo.

In [ ]:
df = rename_columns(df)

print("Colonne rinominate:")
print(df.columns.tolist())

---

## US-04 · Controllo dei Duplicati <a id="us-04"></a>
**User Story**: come Ufficio per il Successo Accademico, vogliamo verificare la presenza di righe duplicate nel dataset, affinché non ci siano studenti contati più volte.

**Responsabile**: Jessica M Pucci

### Obiettivo
Verificare la presenza di righe duplicate nel dataset a valle della rimozione degli Enrolled (§ 2) e della rinomina delle variabili (§ 3), e documentare la decisione su come gestirle.

Una riga duplicata, in questo contesto, rappresenterebbe due studenti con valori identici su tutte le 37 variabili: stesso corso, stessa età, stessi voti semestrali, stessi indicatori socio-economici, stesso outcome.

In [ ]:
# Calcolo delle metriche di base
n_total = len(df)
n_duplicates = df.duplicated().sum()  # True dalla 2a occorrenza in poi
pct_duplicates = (n_duplicates / n_total) * 100

# Output leggibile
print(f"Righe totali:    {n_total}")
print(f"Righe duplicate: {n_duplicates} ({pct_duplicates:.2f}%)")

# Rimozione condizionale: agiamo solo se ci sono effettivamente duplicati.
# Rende esplicito l'intento e produce un output chiaro in entrambi i casi.
if n_duplicates > 0:
    # reset_index(drop=True) evita indici "bucati" dopo la rimozione
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Righe dopo rimozione: {len(df)}")
else:
    print("Nessuna rimozione necessaria.")

### Risultato
Il dataset non contiene righe duplicate (**0 su 3.630 righe**, 0,00%).

### Considerazione
Il risultato è coerente con le aspettative. Le variabili sono 36 e la probabilità che due studenti coincidano esattamente su tutte queste dimensioni è trascurabile, salvo errori di inserimento. Inoltre il dataset UCI è già stato preprocessato dagli autori prima della pubblicazione, quindi eventuali duplicati grossolani sarebbero già stati gestiti a monte.

Va comunque sottolineato che il controllo era necessario: la presenza di duplicati avrebbe compromesso la fase di train/test split nello Sprint 3, dove lo stesso individuo rischierebbe di finire in entrambi i set causando **data leakage** — il modello "vedrebbe" in fase di test dati già visti in training, sovrastimando le sue performance.

### Decisione
Nessuna rimozione necessaria. Il dataset mantiene le sue **3.630** righe e si procede con US-05 (Analisi dei Missing Values) sul dataset invariato.